# HV-008 — Dynamic Step Response

**Technical gate:** SR-001  
**Dataset release:** VSM-LAB-DATA-v1.1

This notebook characterizes the DUT response to commanded 30↔70 %RH step transitions.

The analysis separates:

1. command-to-response timing; and
2. DUT lag relative to the measured RH reference.

This prevents chamber dynamics from being misattributed to the DUT.

## 1. Engineering Question

> How quickly and consistently does the RH DUT respond to a commanded step across temperature, direction, and repetition, and how much lag does it exhibit relative to the measured reference trajectory?

Final HVAC suitability is not decided in this notebook.

## 2. Reproducible Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

current_dir = Path.cwd()
candidate_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / "vsm_lab_data_v1_1").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate project root containing "
        "'data/raw/vsm_lab_data_v1_1'."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.step_response_analysis import (
    build_step_metrics_table,
    load_dynamic_step,
)

tables_dir = project_root / "reports" / "tables"
figures_dir = project_root / "reports" / "figures"
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")

## 3. SR-01 — Dynamic-Step Dataset Audit

Expected design:

- 30 Test_IDs;
- 9,030 rows;
- 301 samples/Test_ID;
- 1 Hz sampling;
- 0–300 s elapsed time;
- command transition at 60 s;
- temperatures 15/25/35 °C;
- UP and DOWN directions;
- five repetitions per Temperature × Direction condition.

In [ ]:
step_df, test_matrix_df = load_dynamic_step(project_root)

step_matrix_df = test_matrix_df.loc[
    test_matrix_df["Test_Family"].astype(str).str.upper().eq("DYNAMIC_STEP")
].copy()

group_sizes = step_df.groupby("Test_ID").size()

print("Dynamic-step dataset audit")
print("--------------------------")
print(f"Measurement rows      : {len(step_df)}")
print(f"Measurement Test_IDs  : {step_df['Test_ID'].nunique()}")
print(f"Matrix Test_IDs       : {step_matrix_df['Test_ID'].nunique()}")
print(f"Samples/Test_ID       : {sorted(group_sizes.unique().tolist())}")
print(
    "Temperatures C       : "
    f"{sorted(step_df['Temperature_Setpoint_C'].unique().tolist())}"
)
print(
    "Directions           : "
    f"{sorted(step_df['Direction'].astype(str).str.upper().unique().tolist())}"
)
print(
    "Repetitions          : "
    f"{sorted(step_df['Repetition'].unique().tolist())}"
)

assert len(step_df) == 9030
assert step_df["Test_ID"].nunique() == 30
assert step_matrix_df["Test_ID"].nunique() == 30
assert sorted(group_sizes.unique().tolist()) == [301]
assert sorted(step_df["Temperature_Setpoint_C"].unique().tolist()) == [15.0, 25.0, 35.0]
assert set(step_df["Direction"].astype(str).str.upper().unique()) == {"UP", "DOWN"}
assert sorted(step_df["Repetition"].unique().tolist()) == [1, 2, 3, 4, 5]

print("\nSR-01 dataset audit: PASS")

## 4. SR-02 — Command Transition Verification

The transition time is detected from `RH_Command`, not assumed.

Expected design value:

```text
t_command = 60 s
```

In [ ]:
transition_rows = []

for test_id, group in step_df.groupby("Test_ID"):
    g = group.sort_values("Elapsed_s")
    initial_command = float(g["RH_Command"].iloc[0])
    changed = ~np.isclose(
        g["RH_Command"].to_numpy(dtype=float),
        initial_command,
        rtol=0.0,
        atol=1e-12,
    )
    idx = np.flatnonzero(changed)
    if idx.size == 0:
        raise ValueError(f"{test_id}: no command transition detected.")
    t_command = float(g["Elapsed_s"].iloc[idx[0]])

    transition_rows.append(
        {
            "Test_ID": test_id,
            "Temperature_Setpoint_C": float(g["Temperature_Setpoint_C"].iloc[0]),
            "Direction": str(g["Direction"].iloc[0]).upper(),
            "Repetition": int(g["Repetition"].iloc[0]),
            "Initial_Command": initial_command,
            "Final_Command": float(g["RH_Command"].iloc[-1]),
            "t_command_s": t_command,
        }
    )

transition_df = pd.DataFrame(transition_rows)

display(transition_df.head())

assert transition_df["t_command_s"].nunique() == 1
assert np.isclose(transition_df["t_command_s"].iloc[0], 60.0)

print("\nSR-02 command transition verification: PASS")

## 5. SR-03 — Per-Test Step Metrics

For each Test_ID, the notebook extracts the full metric set separately for the reference and DUT:

- t10 / operational dead time;
- first-order-equivalent τ at 63.2%;
- t90;
- ±2% settling time;
- overshoot;
- DUT-minus-reference lag at t10, τ, and t90.

Initial and final levels use robust median plateau estimates.

In [ ]:
step_metrics_df = build_step_metrics_table(step_df)

print(f"Metric rows: {len(step_metrics_df)}")
assert len(step_metrics_df) == 30
assert step_metrics_df["Test_ID"].nunique() == 30
assert np.allclose(step_metrics_df["t_command_s"], 60.0)

display(
    step_metrics_df[
        [
            "Test_ID",
            "Temperature_Setpoint_C",
            "Direction",
            "Repetition",
            "Reference_dead_time_s",
            "DUT_dead_time_s",
            "Reference_tau_s",
            "DUT_tau_s",
            "Reference_t90_s",
            "DUT_t90_s",
            "Lag_t10_s",
            "Lag_tau_s",
            "Lag_t90_s",
            "DUT_settling_time_s",
            "DUT_overshoot_pct",
        ]
    ].head(10).round(3)
)

print("\nSR-03 per-Test_ID metric extraction: PASS")

## 6. SR-04 — Metric Completeness Audit

A missing settling-time value is retained as NaN if the response does not enter and remain inside the ±2% band before the record ends. Other threshold metrics should normally be finite.

In [ ]:
metric_columns = [
    "Reference_dead_time_s",
    "DUT_dead_time_s",
    "Reference_tau_s",
    "DUT_tau_s",
    "Reference_t90_s",
    "DUT_t90_s",
    "Lag_t10_s",
    "Lag_tau_s",
    "Lag_t90_s",
]

completeness = pd.DataFrame(
    {
        "Metric": metric_columns
        + ["Reference_settling_time_s", "DUT_settling_time_s"],
        "N_Finite": [
            int(np.isfinite(step_metrics_df[col]).sum())
            for col in metric_columns
        ]
        + [
            int(np.isfinite(step_metrics_df["Reference_settling_time_s"]).sum()),
            int(np.isfinite(step_metrics_df["DUT_settling_time_s"]).sum()),
        ],
    }
)
completeness["N_Total"] = len(step_metrics_df)
completeness["N_Missing"] = (
    completeness["N_Total"] - completeness["N_Finite"]
)

display(completeness)

assert (completeness.loc[
    completeness["Metric"].isin(metric_columns),
    "N_Missing",
] == 0).all()

print("\nSR-04 metric completeness audit: PASS")

## 7. SR-05 — Condition-Level Summary

Five repetitions are available at each Temperature × Direction condition.

The condition summary reports mean, SD, median, min, and max for the main dynamic metrics.

In [ ]:
summary_metrics = [
    "Reference_dead_time_s",
    "DUT_dead_time_s",
    "Reference_tau_s",
    "DUT_tau_s",
    "Reference_t90_s",
    "DUT_t90_s",
    "Lag_t10_s",
    "Lag_tau_s",
    "Lag_t90_s",
    "Reference_settling_time_s",
    "DUT_settling_time_s",
    "Reference_overshoot_pct",
    "DUT_overshoot_pct",
]

condition_rows = []
for (temp, direction), group in step_metrics_df.groupby(
    ["Temperature_Setpoint_C", "Direction"]
):
    row = {
        "Temperature_Setpoint_C": temp,
        "Direction": direction,
        "N": len(group),
    }
    for metric in summary_metrics:
        values = group[metric]
        row[f"{metric}_Mean"] = values.mean()
        row[f"{metric}_SD"] = values.std(ddof=1)
        row[f"{metric}_Median"] = values.median()
        row[f"{metric}_Min"] = values.min()
        row[f"{metric}_Max"] = values.max()
    condition_rows.append(row)

condition_summary_df = pd.DataFrame(condition_rows).sort_values(
    ["Temperature_Setpoint_C", "Direction"]
).reset_index(drop=True)

display(
    condition_summary_df[
        [
            "Temperature_Setpoint_C",
            "Direction",
            "N",
            "Reference_t90_s_Mean",
            "DUT_t90_s_Mean",
            "Lag_t90_s_Mean",
            "DUT_t90_s_SD",
            "DUT_settling_time_s_Mean",
            "DUT_overshoot_pct_Mean",
        ]
    ].round(3)
)

## 8. SR-06 — Overall Direction and Temperature Diagnostics

In [ ]:
direction_summary_df = (
    step_metrics_df.groupby("Direction", as_index=False)
    .agg(
        N=("Test_ID", "size"),
        Reference_t90_Mean_s=("Reference_t90_s", "mean"),
        DUT_t90_Mean_s=("DUT_t90_s", "mean"),
        Lag_t90_Mean_s=("Lag_t90_s", "mean"),
        Lag_t90_SD_s=("Lag_t90_s", "std"),
        DUT_Settling_Mean_s=("DUT_settling_time_s", "mean"),
        DUT_Overshoot_Mean_pct=("DUT_overshoot_pct", "mean"),
    )
)

temperature_summary_df = (
    step_metrics_df.groupby("Temperature_Setpoint_C", as_index=False)
    .agg(
        N=("Test_ID", "size"),
        Reference_t90_Mean_s=("Reference_t90_s", "mean"),
        DUT_t90_Mean_s=("DUT_t90_s", "mean"),
        Lag_t90_Mean_s=("Lag_t90_s", "mean"),
        Lag_t90_SD_s=("Lag_t90_s", "std"),
        DUT_Settling_Mean_s=("DUT_settling_time_s", "mean"),
        DUT_Overshoot_Mean_pct=("DUT_overshoot_pct", "mean"),
    )
)

print("By Direction")
display(direction_summary_df.round(3))

print("By Temperature")
display(temperature_summary_df.round(3))

## 9. Curated HV-008 Evidence Export

In [ ]:
step_metrics_df.to_csv(
    tables_dir / "hv008_step_test_metrics.csv",
    index=False,
)
condition_summary_df.to_csv(
    tables_dir / "hv008_step_summary_by_condition.csv",
    index=False,
)

overall_summary_df = pd.concat(
    [
        direction_summary_df.assign(Summary_Type="Direction"),
        temperature_summary_df.rename(
            columns={"Temperature_Setpoint_C": "Direction"}
        ).assign(Summary_Type="Temperature"),
    ],
    ignore_index=True,
    sort=False,
)
overall_summary_df.to_csv(
    tables_dir / "hv008_step_overall_summary.csv",
    index=False,
)

print("Curated HV-008 tables exported.")

## 10. Initial SR-001 Checkpoint

A clean-kernel run should establish:

- SR-01 dataset audit PASS;
- SR-02 command-transition verification PASS;
- SR-03 per-Test_ID metric extraction PASS;
- SR-04 metric completeness audit PASS;
- condition-level summaries across all six Temperature × Direction cells;
- overall Direction and Temperature summaries;
- curated HV-008 CSV exports.

The next step is engineering interpretation of the extracted response metrics, followed by representative figures and formal SR-001 gate closure.